# Sprint 4 — Baseline Implementation & Reproducible Code
## Від Experimental Protocol до перевірюваної програмної реалізації

**AI Research Practices · Науково-дослідна практика · Sprint 4 із 7**

**Вхід:** `Research_Proposal_v1.md`, `Experimental_Protocol_v1.md`, гіпотеза H1, RQ1–RQ3 та матриця експериментів зі Sprint 3.

**Мета:** реалізувати підготовку даних, навчання й коректне налаштування моделей, збереження артефактів та відтворюваний механізм тестового шуму, і провести пілотне вимірювання для перевірки працездатності всього експериментального ланцюжка.

**Результат:** **Reproducible Baseline v1.0** — код, конфігурації, зафіксовані моделі, перевірки коректності, пілотні `Q(m,σ)` і `D(m,σ)` та `Implementation_Report_v1.md` у персональному GitHub fork.

**Межа Sprint 4:** створюємо та технічно перевіряємо реалізацію протоколу, включно з **одним пілотним шумовим вимірюванням** для всіх трьох моделей. **Не аналізуємо профілі деградації й не робимо висновків про H1** — основні вимірювання за всією матрицею заплановані на Sprint 5.

---

**Наскрізний приклад:** Logistic Regression (Baseline Model), Random Forest і Gradient Boosting; навчання без штучного шуму; добір гіперпараметрів тільки на Training Set; адитивний гаусівський шум **лише в стандартизованих тестових ознаках**. За Sprint 3: $\sigma\in\{0,0.1,0.2,0.4,0.6\}$, macro-F1, п'ять заздалегідь визначених seed для кожного додатного $\sigma$.

**Важливо щодо даних:** у демонстраційному коді використано вбудований датасет `load_wine` бібліотеки scikit-learn. Він слугує **навчальним замінником** датасету з числовими ознаками, а не даними датчиків обладнання. Для власного дослідження обираємо дані, передбачені погодженим протоколом.

## Маршрут спринту

1. Перетворюємо Experimental Protocol на послідовність програмних операцій та перевірок.
2. Фіксуємо дані, поділ, генератори випадкових чисел і залежності.
3. Реалізуємо препроцесинг без data leakage.
4. Підбираємо гіперпараметри через Cross-Validation та навчаємо три фіксовані моделі.
5. Вимірюємо чисту якість `Q(m,0)` і тестуємо генератор шуму.
6. Виконуємо пілот: одна реалізація шуму для трьох моделей, розрахунок `Q(m,0.2)` і `D(m,0.2)`.
7. Зберігаємо артефакти, перевіряємо відтворюваність і готуємо передачу до Sprint 5.

**Ключова навичка:** зробити так, щоб інший дослідник міг повторити **ту саму процедуру**, а не лише запустити схожий код.

## A. Problem Situation — чи можна відтворити наш експеримент?

У Sprint 3 сплановано 15 експериментальних умов і 63 оцінювання macro-F1. Але сам план не гарантує правильного виконання: достатньо один раз стандартизувати весь датасет перед поділом, додати шум до неправильної шкали або переналаштувати модель після тестування — і отримані результати вже не відповідатимуть протоколу.

З'ясуємо:

- як забезпечити незалежність Test Set під час tuning;
- як застосувати стандартизацію **до** внесення тестового шуму;
- як використати **одну й ту саму реалізацію шуму** для всіх моделей у парному порівнянні;
- як перевірити, що моделі не перенавчаються між шумовими умовами;
- як провести одне пілотне вимірювання `D(m,σ)` та перевірити весь шлях від тестових даних до результату;
- що зберегти, щоб дослід можна було відтворити.

**Проблема спринту:** відтворюваність потребує не лише збереженого notebook, а й явних правил підготовки даних, настроювання, випадковості та збереження артефактів.

## B1. Essential Theory — від протоколу до перевірюваного коду

Протокол зі Sprint 3 задає обмеження на **допустимі дії програми**. Перекладаємо їх на перевірки:

| Вимога протоколу | Що реалізуємо | Що перевіряємо |
|---|---|---|
| Один Training/Test split | Зафіксовані індекси й seed поділу | Немає перетину індексів |
| Немає data leakage | `StandardScaler` всередині `Pipeline` і CV | Масштабування навчається лише на train/fold |
| Tuning без Test Set | `GridSearchCV` тільки на Training Set | Тест не передається у `fit` / `search` |
| Чисте навчання | Обрані моделі навчаємо без штучного шуму | Шум не присутній у навчальних ознаках |
| Спільні шумові умови | Одна матриця шуму для однакового `σ`, seed | Ідентичний шум для всіх моделей |
| Пілотне вимірювання | Одна визначена пара (`σ=0.2`, `seed=11`) для всіх моделей | `Q`, `D`, спільний шум, відтворення результату |
| Відтворюваність | Конфігурації, версії, seed, файли моделей | Повторний запуск за тим самим середовищем |

Технічні перевірки виявляють помилки **реалізації**; вони не доводять правильність H1 та не замінюють наукової інтерпретації.

## B2. Essential Theory — Pipeline, Cross-Validation та відсутність витоку даних

Під час навчання класифікатора часто виконуємо кілька послідовних операцій: спочатку підготовку числових ознак, потім навчання алгоритму. Бібліотека **scikit-learn** дозволяє об'єднати їх в об'єкт **Pipeline** — єдину послідовність кроків.

**`Pipeline(StandardScaler → Classifier)`** містить два етапи:

1. **`StandardScaler`** за навчальними даними обчислює середнє значення та стандартне відхилення кожної ознаки й стандартизує значення.
2. **`Classifier`** отримує стандартизовані ознаки, навчається розпізнавати класи та прогнозує їх для нових даних.

Під час виклику `pipeline.fit(X_train, y_train)` спочатку налаштовуємо `StandardScaler`, а потім навчаємо класифікатор. Виклик `pipeline.predict(X_test)` застосовує **раніше визначену стандартизацію** до тестових даних і передає їх навченому класифікатору.

**Чому Pipeline потрібен під час Cross-Validation?** Навчальні дані послідовно розбиваємо на навчальні й валідаційні частини. Якщо стандартизувати весь Training Set заздалегідь, параметри масштабування врахують і поточну валідаційну частину — виникне **Data Leakage**. Якщо `StandardScaler` розміщений усередині `Pipeline`, на кожній складці Cross-Validation його параметри обчислюються **лише на навчальній частині цієї складки**.

**Hyperparameter Tuning** виконуємо через `GridSearchCV`: перевіряємо заздалегідь визначені конфігурації за `macro-F1` на чистих валідаційних даних. Отримуємо найкращі знайдені конфігурації *в межах пошуку*, а не гарантовано глобальний оптимум. Наприкінці `GridSearchCV(refit=True)` повторно навчає обрану `Pipeline` на всьому Training Set.

**Test Set не використовуємо для вибору конфігурацій або для визначення параметрів стандартизації.** Для нашого шумового експерименту є додаткова особливість: шум вносимо *після стандартизації*, тому шлях прогнозування для зашумлених ознак реалізуємо окремо у B3 та C6.

## B3. Essential Theory — стандартизований шум та фіксована модель

У Sprint 3 визначено:

$$
 z=\frac{x-\mu}{s},\qquad \varepsilon\sim\mathcal{N}(0,\sigma^2),\qquad z_{\mathrm{noisy}}=z_{\mathrm{clean}}+\varepsilon.
$$

Тут $\mu$ і $s$ визначено **лише на Training Set**. Оскільки $\sigma$ задано в одиницях **стандартизованих** ознак, шум потрібно додавати **після** `StandardScaler` і **перед** класифікатором.

Звичайний виклик `fitted_pipeline.predict(raw_test)` коректний для чистих даних, але виклик `fitted_pipeline.predict(z_noisy)` **помилково стандартизує їх повторно**. У нашій навчальній реалізації тому явно розділяємо вже навчений scaler та classifier:

**Raw Test Set → fitted scaler.transform → додавання шуму → fitted classifier.predict.**

Змінюються тільки $\sigma$ і реалізація шуму. Модель, Test Set, спосіб перетворення та метрика залишаються незмінними. Для кожної пари ($\sigma$, seed) генеруємо **одну спільну матрицю шуму** для всіх алгоритмів.

## B4. Essential Theory — відтворюваність, перевірки та розподіл робіт

Фіксуємо: походження й версію датасету; індекси Training/Test; схему Cross-Validation; простори tuning; seed для поділу, моделей і шуму; версії Python та бібліотек; збережені моделі; інструкцію повторного запуску.

**Random seed** задає початковий стан генератора псевдовипадкових чисел. Різні seed за одного $\sigma$ створюють різні реалізації шуму; той самий seed за однакових умов дозволяє відтворити реалізацію.

**Smoke test** перевіряє технічну працездатність окремого компонента (наприклад, форму й відтворюваність шумової матриці). **Pilot Experiment** перевіряє весь ланцюжок на одній заздалегідь визначеній умові: `σ=0.2`, `seed=11`, для трьох моделей із розрахунком `Q` та `D`. **Main Experiment** у Sprint 5 охоплює повну матрицю, повторення, невизначеність та H1.

Оскільки пілот використовує Test Set, його результати **не слугують підставою** для tuning, зміни метрики, рівнів шуму або вибору бажаного результату. За відсутності технічних помилок ці три оцінки `Q(m,0.2)` можуть увійти до 63 запланованих оцінювань Sprint 5 без повторного додавання як нових вимірювань. Якщо виправлено технічну помилку, документуємо її та повторюємо всі зачеплені вимірювання.

Навчальні блоки коду нижче є прикладом **технічної реалізації**. Повні серії вимірювань і висновки щодо RQ1–RQ3 залишаються за межами цього спринту.

## C1. Guided Research Example — реалізуємо конфігурацію протоколу

Визначимо параметри в одному місці. У Sprint 3 зафіксували п'ять рівнів шуму й п'ять seed для додатних рівнів. У Sprint 4 **не проходимо всю матрицю**: перевіримо генератор на синтетичній матриці, а потім виконаємо **одне пілотне вимірювання** (`σ=0.2`, `seed=11`) для всіх моделей без розрахунку профілів деградації.

Нижченаведений код виконуємо **послідовно, від першої кодової комірки**. Для запуску потрібні `numpy`, `pandas`, `scikit-learn`, `joblib`; результати tuning можуть залежати від версій пакетів. Код не завантажує даних із мережі.

In [ ]:
from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd
import sklearn
import joblib

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import f1_score

# Єдина конфігурація даних, tuning і шуму — для відтворення запуску.
CONFIG = {
    "dataset": "sklearn.datasets.load_wine",
    "split_seed": 42,
    "model_seed": 42,
    "test_size": 0.20,
    "cv_folds": 3,
    "cv_score": "f1_macro",
    "noise_levels": [0.0, 0.1, 0.2, 0.4, 0.6],
    "noise_seeds": [11, 23, 37, 41, 53],
    "pilot_sigma": 0.2,
    "pilot_seed": 11,
}
ARTIFACT_DIR = Path("research_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)
print("Конфігурацію визначено:", CONFIG)

## C2. Guided Research Example — завантажуємо дані та фіксуємо split

Для повторюваного прикладу використовуємо невеликий реальний навчальний датасет вин трьох класів. Він не імітує дані датчиків, тому результати на ньому не можна переносити на моніторинг обладнання.

Розбиваємо індекси **до** стандартизації; індекси зберігаємо. Стратифікація допомагає зберегти пропорції класів. Якщо у власному дослідженні є часові або групові залежності, потрібне інше правило split — відповідно до Experimental Protocol.

In [ ]:
# Завантажуємо дані й один раз визначаємо незалежний Test Set.
raw = load_wine(as_frame=True)
X = raw.data.copy()
y = raw.target.copy()
all_idx = np.arange(len(X))

train_idx, test_idx = train_test_split(
    all_idx,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["split_seed"],
    stratify=y,
)
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

assert set(train_idx).isdisjoint(set(test_idx)), "Train/Test перетинаються"
assert len(train_idx) + len(test_idx) == len(X)
assert X_train.columns.tolist() == X_test.columns.tolist()

np.savez(ARTIFACT_DIR / "split_indices.npz", train=train_idx, test=test_idx)
print("Форма X_train / X_test:", X_train.shape, X_test.shape)
print("Класи у Training Set:", y_train.value_counts().sort_index().to_dict())

## C3. Guided Research Example — tuning трьох моделей без Test Set

Для кожного алгоритму задаємо окремий **обмежений простір пошуку**. Використовуємо однакові складки та ту саму метрику `f1_macro`. Конфігурації не мають однакових числових гіперпараметрів — алгоритми різні.

`GridSearchCV(refit=True)` після порівняння конфігурацій навчає найкращу знайдену `Pipeline` на **всьому Training Set**. Це і є остаточна модель, яку надалі фіксуємо. `X_test` та `y_test` до tuning не передаємо.

Сітки невеликі для навчального запуску; у реальному дослідженні простори пошуку й обчислювальний бюджет обґрунтовуємо до виконання.

In [ ]:
# Ті самі складки Cross-Validation для всіх алгоритмів.
cv = StratifiedKFold(
    n_splits=CONFIG["cv_folds"],
    shuffle=True,
    random_state=CONFIG["split_seed"],
)

specs = {
    "LR": (
        LogisticRegression(max_iter=2000, random_state=CONFIG["model_seed"]),
        {"model__C": [0.1, 1.0, 10.0]},
    ),
    "RF": (
        RandomForestClassifier(random_state=CONFIG["model_seed"], n_jobs=1),
        {"model__n_estimators": [80, 160], "model__max_depth": [None, 6]},
    ),
    "GB": (
        GradientBoostingClassifier(random_state=CONFIG["model_seed"]),
        {"model__n_estimators": [60, 120], "model__learning_rate": [0.05, 0.1]},
    ),
}

# Підбираємо гіперпараметри тільки на Training Set; refit навчає фінальну Pipeline.
fitted_models = {}
tuning_log = {}
for name, (estimator, param_grid) in specs.items():
    pipe = Pipeline([("scale", StandardScaler()), ("model", estimator)])
    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring=CONFIG["cv_score"],
        cv=cv,
        refit=True,
        n_jobs=1,
    )
    search.fit(X_train, y_train)
    fitted_models[name] = search.best_estimator_
    tuning_log[name] = {
        "best_params": search.best_params_,
        "best_cv_macro_f1": float(search.best_score_),
        "candidate_count": len(search.cv_results_["params"]),
    }
    print(name, "CV macro-F1:", round(search.best_score_, 4),
          "параметри:", search.best_params_)

## C4. Guided Research Example — фіксуємо чисті моделі та `Q(m,0)`

Після tuning **не змінюємо** конфігурації моделей на основі Test Set. Для кожного алгоритму одноразово оцінюємо `Q(m,0)` — **Clean Reference Performance**, тобто якість без штучного шуму. Це не те саме, що **Baseline Model**: Logistic Regression — опорний алгоритм міжмодельного порівняння, а `Q(m,0)` є власним базовим рівнем якості *кожної* моделі.

У цьому спринті отримане значення — технічно необхідна чиста оцінка для майбутнього обчислення `D(m,σ)`. **Не порівнюємо деградацію й не перевіряємо H1** до виконання повної матриці Sprint 5.

In [ ]:
# Оцінюємо чисту якість на Test Set та зберігаємо фіксовані моделі.
clean_reference = {}
for name, fitted in fitted_models.items():
    pred = fitted.predict(X_test)
    clean_reference[name] = float(f1_score(y_test, pred, average="macro"))
    joblib.dump(fitted, ARTIFACT_DIR / f"model_{name}.joblib")

print("Clean Reference Performance, Q(m,0):")
for name, score in clean_reference.items():
    print(f"  {name}: {score:.4f}")

## C5. Guided Research Example — генератор тестового шуму та його технічні перевірки

Визначимо функцію, яка приймає **вже стандартизовані** ознаки й повертає нову матрицю з адитивним незалежним гаусівським шумом. Виклик із фіксованими $\sigma$ та seed має бути відтворюваним. За $\sigma=0$ вхідні значення не змінюються.

Використаємо **синтетичну тестову матрицю**, а не фактичний Test Set: так ми перевіримо механізм, не починаючи основних вимірювань і не створюючи підстав підлаштовувати моделі за тестовими результатами.

In [ ]:
# Генеруємо відтворюване спотворення стандартизованих ознак.
def add_gaussian_noise(standardized_features, sigma: float, seed: int):
    """Додає N(0, sigma^2) до стандартизованих числових ознак."""
    array = np.asarray(standardized_features, dtype=float)
    if array.ndim != 2 or not np.isfinite(array).all():
        raise ValueError("Очікуємо скінченну двовимірну матрицю")
    if sigma < 0 or not np.isfinite(sigma):
        raise ValueError("sigma має бути невід'ємним скінченним числом")
    rng = np.random.default_rng(seed)
    noise = rng.normal(loc=0.0, scale=sigma, size=array.shape)
    return array + noise

# Перевіряємо форму, seed, σ=0 та незмінність вихідної матриці.
sample = np.zeros((4, 3))
a = add_gaussian_noise(sample, sigma=0.2, seed=11)
b = add_gaussian_noise(sample, sigma=0.2, seed=11)
c = add_gaussian_noise(sample, sigma=0.2, seed=23)

assert a.shape == sample.shape
assert np.array_equal(a, b), "Один seed не відтворює шум"
assert not np.array_equal(a, c), "Різні seed дали однакову реалізацію"
assert np.array_equal(sample, add_gaussian_noise(sample, 0.0, 11))
assert np.array_equal(sample, np.zeros_like(sample)), "Вхідну матрицю змінено"
print("Smoke tests генератора шуму пройдено")

## C6. Guided Research Example — маршрут шуму без подвійної стандартизації

Після завершення tuning кожна навчена `Pipeline` містить `scale` та `model`. Для шумових експериментів **не передаємо стандартизовані значення у `pipeline.predict()`**, інакше scaler спрацює вдруге.

Підготуємо функцію, що визначає правильний шлях обробки: scaler кожної фіксованої моделі застосовуємо до початкового Test Set, додаємо шум і звертаємося безпосередньо до класифікатора. У навчальному прикладі всі scaler навчені на тому самому Training Set; при парному порівнянні переконуємося, що їхні параметри однакові, щоб застосувати спільну матрицю шуму.

**У Sprint 4 виконуємо тільки пілотну умову.** Повну матрицю `Q(m,σ)` і `D(m,σ)` опрацюємо у Sprint 5.

In [ ]:
# Для даного прикладу всі три scaler вчилися на однаковому Training Set.
reference_scaler = fitted_models["LR"].named_steps["scale"]
for name, pipe in fitted_models.items():
    scaler = pipe.named_steps["scale"]
    assert np.allclose(scaler.mean_, reference_scaler.mean_)
    assert np.allclose(scaler.scale_, reference_scaler.scale_)

# Стандартизуємо один раз: шум додамо у пілоті після цієї операції.
def predict_from_standardized(fitted_pipeline, standardized_features):
    """Класифікація вже стандартизованих ознак без повторного scale."""
    return fitted_pipeline.named_steps["model"].predict(standardized_features)

# Технічна перевірка на CLEAN Test Set: два шляхи повинні збігатися.
z_test_clean = reference_scaler.transform(X_test)
for name, pipe in fitted_models.items():
    assert np.array_equal(
        pipe.predict(X_test),
        predict_from_standardized(pipe, z_test_clean),
    ), f"Розбіжність у маршруті препроцесингу: {name}"
print("Маршрут прогнозування без подвійної стандартизації перевірено")

## C7. Guided Research Example — Pilot Experiment: перевіряємо весь ланцюжок

Виконаємо одну **заздалегідь визначену** умову зі Sprint 3: $\sigma_{\mathrm{pilot}}=0.2$, `seed=11`. Для кожної моделі обчислили її чистий рівень якості `Q(m,0)` у C4. Тепер **один раз** додаємо шум до спільної стандартизованої матриці Test Set й обчислюємо `Q(m,0.2)` та `D(m,0.2)=Q(m,0)-Q(m,0.2)`.

Для всіх трьох моделей використовуємо **ті самі зашумлені тестові ознаки**, але кожна модель має власну чисту оцінку `Q(m,0)`. Перевіряємо відтворюваність повторним обчисленням із тим самим seed; не перенавчаємо моделі.

**Це технічний пілот**, а не перевірка H1: одна реалізація шуму не дозволяє оцінити профілі деградації, варіативність або статистичну переконливість відмінностей. За результатами пілоту не змінюємо tuning і протокол. У Sprint 5 доповнюємо інші комірки матриці й повторення; уже отриманий коректний результат враховуємо **без подвійного підрахунку**.

In [ ]:
# Одна зафіксована пілотна умова і спільна матриця шуму для всіх моделей.
pilot_sigma = CONFIG["pilot_sigma"]
pilot_seed = CONFIG["pilot_seed"]
assert pilot_sigma in CONFIG["noise_levels"] and pilot_seed in CONFIG["noise_seeds"]
z_test_pilot = add_gaussian_noise(z_test_clean, pilot_sigma, pilot_seed)
assert np.array_equal(z_test_pilot, add_gaussian_noise(z_test_clean, pilot_sigma, pilot_seed))
assert z_test_pilot.shape == z_test_clean.shape

# Оцінюємо Q та D, звертаючись до classifier без повторного StandardScaler.
pilot_results = []
for name, pipe in fitted_models.items():
    prediction = predict_from_standardized(pipe, z_test_pilot)
    q_noise = float(f1_score(y_test, prediction, average="macro"))
    q_clean = clean_reference[name]
    pilot_results.append({
        "model": name,
        "sigma": pilot_sigma,
        "seed": pilot_seed,
        "Q_clean": q_clean,
        "Q_noisy": q_noise,
        "D": q_clean - q_noise,
    })

pilot_table = pd.DataFrame(pilot_results).set_index("model")
assert np.isfinite(pilot_table[["Q_clean", "Q_noisy", "D"]].to_numpy()).all()
assert np.allclose(pilot_table["Q_clean"] - pilot_table["Q_noisy"], pilot_table["D"])
print("Pilot Experiment — технічна перевірка, не висновок про H1:")
print(pilot_table.round(4).to_string())


## C8. Guided Research Example — зберігаємо артефакти та спосіб повторення

Фіксуємо параметри і версії пакетів, результати tuning (це **валідаційні оцінки**, не висновки щодо H1), чисті оцінки, пілотні результати та індекси split. Серіалізовані моделі `joblib` завантажуємо **лише з довірених джерел**, оскільки формати на основі pickle не призначені для безпечного обміну з невідомими авторами.

Для повного відтворення необхідно зберегти також файли вихідних даних або спосіб отримання їхньої точної версії. У нашому прикладі джерело — `sklearn.datasets.load_wine`.

In [ ]:
# Зберігаємо конфігурації, моделі й результати пілоту для незалежної перевірки.
pilot_table.to_csv(ARTIFACT_DIR / "pilot_results.csv", float_format="%.10g")
manifest = {
    "sprint": 4,
    "dataset": CONFIG["dataset"],
    "dataset_rows": len(X),
    "feature_names": X.columns.tolist(),
    "config": CONFIG,
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "sklearn_version": sklearn.__version__,
    "joblib_version": joblib.__version__,
    "tuning": tuning_log,
    "clean_reference_macro_f1": clean_reference,
    "pilot_results": pilot_results,
    "pilot_results_file": "pilot_results.csv",
    "model_files": [f"model_{name}.joblib" for name in fitted_models],
    "split_indices_file": "split_indices.npz",
    "status": "pilot_passed_main_experiment_not_yet_analyzed",
}
(ARTIFACT_DIR / "manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Артефакти:", sorted(p.name for p in ARTIFACT_DIR.iterdir()))

## C9. Guided Research Example — контроль готовності до Sprint 5

До вимірювань за Experimental Matrix перевіримо відповідність реалізації умовам Sprint 3.

| Вимога | Доказ у реалізації |
|---|---|
| Поділ без перетину | `split_indices.npz`, перевірка індексів |
| Tuning тільки на train | `GridSearchCV.fit(X_train, y_train)` |
| Масштабування всередині CV | `Pipeline(scale, model)` |
| Навчання без доданого шуму | `fit` отримує чистий Training Set |
| Фіксовані моделі | Збережені `model_*.joblib` |
| Чисті рівні якості | `clean_reference_macro_f1` у `manifest.json` |
| Генератор шуму | `add_gaussian_noise` та тести на синтетичних даних |
| Пілот Q і D | `pilot_results.csv`, `pilot_results` у `manifest.json`; одна умова для трьох моделей |
| Вірний порядок перетворень | `predict_from_standardized` і перевірка проти `pipeline.predict()` |
| Спільна реалізація шуму | У пілоті одна матриця `z_test_pilot` для всіх моделей |

Отриманий код **перевірено пілотним вимірюванням**, але воно саме по собі ще не відповідає на RQ1–RQ3. Порівняльні вимірювання та оцінку невизначеності виконаємо у Sprint 5.

## Тренувальна вправа 1 — знайдемо data leakage

**Ситуація:** перед поділом даних розробник виконав `StandardScaler().fit_transform(X)` для всього датасету, потім розділив стандартизовані значення на Training Set та Test Set.

1. Визначаємо, які відомості з Test Set використані передчасно.
2. Пропонуємо правильний порядок операцій.
3. Пояснюємо, як `Pipeline` допомагає під час Cross-Validation.

Відповідь для самоперевірки наведено в розділі G після контрольних питань.

## Тренувальна вправа 2 — виявимо подвійне масштабування

**Ситуація:** після `z_test = fitted_pipeline.named_steps["scale"].transform(X_test)` додаємо шум `z_noisy = z_test + noise`, а потім помилково викликаємо `fitted_pipeline.predict(z_noisy)`.

1. Визначаємо, яка операція буде виконана повторно.
2. Записуємо правильну послідовність перетворень.
3. Пояснюємо, чому шум потрібно додавати після стандартизації саме в нашому протоколі.

Відповідь для самоперевірки наведено в розділі G після контрольних питань.

## Тренувальна вправа 3 — перевіримо відтворюваність

**Ситуація:** для одного рівня шуму виконавець щоразу створює генератор без зафіксованого seed. При цьому результати різних моделей отримано з різними матрицями шуму.

1. Визначаємо два недоліки організації порівняння.
2. Пропонуємо спосіб отримати п'ять відтворюваних реалізацій.
3. Пояснюємо, як забезпечити парність порівнянь між трьома моделями.
4. Визначаємо, чи доводять п'ять реалізацій узагальнюваність результатів на інші датасети.

Відповідь для самоперевірки наведено в розділі G після контрольних питань.

## D. Independent Research — реалізуємо власний Experimental Protocol

Повертаємося до **власного**, погодженого з керівником `Experimental_Protocol_v1.md`. Навчальний приклад C1–C9 не визначає обов'язковий набір алгоритмів, датасет чи спосіб втручання для інших тем.

Для власного проєкту:

1. Переносимо фактори, метрики, контрольовані умови та план повторень із протоколу в конфігурацію коду.
2. Реалізуємо завантаження даних і фіксуємо точне джерело / версію.
3. Виконуємо поділ і захист від витоку даних відповідно до природи датасету.
4. Реалізуємо та документуємо tuning або іншу процедуру вибору параметрів, визначену в протоколі.
5. Створюємо baseline та інші необхідні фіксовані моделі / процедури.
6. Пишемо технічні тести для генератора експериментального впливу, метрик і відтворюваності.
7. Виконуємо **одне пілотне вимірювання** за заздалегідь визначеною умовою, перевіряємо розрахунок показників і реєструємо результат без інтерпретації гіпотези.
8. Зберігаємо артефакти й інструкції запуску у **власному GitHub fork**.
9. Документуємо розбіжності з протоколом та погоджуємо їх із керівником **до** основних вимірювань.

**Результат:** Reproducible Baseline v1.0, а не експериментальна стаття і не висновок про H1.

### D1. Protocol → Implementation Map

| Вимога з Experimental Protocol | Файл / функція реалізації | Технічна перевірка | Статус |
|---|---|---|---|
| Дані та split | [ ] | [ ] | [ ] |
| Preprocessing | [ ] | [ ] | [ ] |
| Baseline / моделі | [ ] | [ ] | [ ] |
| Налаштування параметрів | [ ] | [ ] | [ ] |
| Експериментальний вплив | [ ] | [ ] | [ ] |
| Метрики / повторення | [ ] | [ ] | [ ] |
| Pilot Experiment | [ ] | [ ] | [ ] |
| Збереження / відтворення | [ ] | [ ] | [ ] |

**Пояснення:** [які вимоги протоколу ще не реалізовано і чому?]

### D2. Дані, поділ і запобігання витоку

**Походження / версія даних та доступ:** [заповнюємо]

**Навчальна й тестова частини та обґрунтування split:** [заповнюємо]

**Препроцесинг, де саме виконується `fit`:** [заповнюємо]

**Перевірки незалежності та цілісності даних:** [заповнюємо]

**Пояснення:** [які помилки можуть призвести до data leakage і як їх виявляємо?]

### D3. Навчання, tuning та фіксація моделей

**Baseline і порівнювані методи:** [заповнюємо]

**Параметри пошуку, критерій, Cross-Validation / інший метод:** [заповнюємо]

**Seed і конфігурації:** [заповнюємо]

**Файли остаточних моделей та спосіб їх завантаження:** [заповнюємо]

**Підтвердження, що Test Set не брав участі в tuning:** [заповнюємо]

**Пояснення:** [чому реалізація відповідає запланованому протоколу?]

### D4. Перевірки механізму експериментального впливу

| Перевірка | Очікувана властивість | Результат технічного тесту |
|---|---|---|
| Форма й тип вихідних даних | [ ] | [ ] |
| Відтворюваність за seed | [ ] | [ ] |
| Незмінність оригінальних даних | [ ] | [ ] |
| Однакові умови між методами | [ ] | [ ] |
| Відповідність шкалі / одиницям впливу | [ ] | [ ] |

### Пілотне вимірювання

**Пілотна умова (значення факторів, seed, обґрунтування):** [заповнюємо]

| Метод / модель | Базова якість | Якість у пілоті | Зміна якості | Перевірка відтворюваності |
|---|---|---|---|---|
| [ ] | [ ] | [ ] | [ ] | [ ] |

**Підсумок технічної перевірки:** [чи коректно працює ланцюжок; чи були помилки або відхилення?]

**Пояснення:** [чому один пілот перевіряє реалізацію, але не доводить H1?]

### D5. Implementation Report і передача до Sprint 5

Заповнюємо `Implementation_Report_v1.md` за шаблоном `Implementation_Report_v1_Template.md`. У персональному fork зберігаємо код, конфігурації, інструкцію запуску, перелік залежностей, перевірки, пілотні показники і **дозволені до публікації** артефакти.

**Планована структура:**

```text
research/
├── proposal/Research_Proposal_v1.md
├── protocol/Experimental_Protocol_v1.md
├── src/                    # реалізація та тести
├── configs/                # параметри і seed
├── models/                 # зафіксовані моделі, якщо дозволено
├── results/                # pilot: Sprint 4; основні вимірювання: Sprint 5
├── Implementation_Report_v1.md
└── README.md               # відтворення запуску
```

**Розбіжності з протоколом:** [описуємо або зазначаємо, що їх немає]

**Питання для керівника до запуску експерименту:** [заповнюємо]

## E1. Sprint Review — перевірка готовності реалізації

### 1. Перевірка навчального прикладу (C1–C9)

Перевіряємо, чи правильно виконано наскрізний приклад і чи зрозуміла логіка його програмної реалізації.

- [ ] Дані завантажено та розділено на Training Set і Test Set.
- [ ] Стандартизацію виконано без Data Leakage, зокрема під час Cross-Validation.
- [ ] Для Logistic Regression, Random Forest і Gradient Boosting виконано Hyperparameter Tuning без використання Test Set.
- [ ] Три моделі навчено, зафіксовано та збережено.
- [ ] Для кожної моделі визначено Clean Reference Performance — `Q(m,0)`.
- [ ] Перевірено генерацію гаусівського шуму та відсутність повторної стандартизації тестових ознак.
- [ ] Проведено Pilot Experiment за `σ=0.2` та `seed=11` зі спільною реалізацією шуму для всіх трьох моделей.
- [ ] Для кожної моделі обчислено `Q(m,0.2)` і `D(m,0.2)`.
- [ ] Перевірено відтворюваність пілотних результатів за однакової конфігурації.

**Результат:** навчальний приклад виконано; продемонстровано працездатність усіх основних компонентів експерименту.

### 2. Перевірка власного дослідницького проєкту

Перевіряємо реалізацію **власного Experimental Protocol v1.0**, погодженого з науковим керівником у Sprint 3.

- [ ] Реалізація відповідає власній гіпотезі, Research Questions та Experimental Matrix.
- [ ] Джерело, версія, спосіб підготовки й поділу даних задокументовані.
- [ ] Реалізовано визначені протоколом моделі, процедури налаштування та контрольовані умови.
- [ ] Перевірено відсутність витоку даних під час навчання, налаштування й оцінювання.
- [ ] Моделі або інші досліджувані процедури підготовлено та зафіксовано для основного експерименту.
- [ ] Реалізовано та перевірено механізм експериментального впливу й обчислення запланованих показників.
- [ ] Виконано одне пілотне вимірювання за умовою, визначеною у власному протоколі.
- [ ] Результати пілоту підтверджують технічну працездатність реалізації; висновки щодо гіпотези поки не формулюємо.
- [ ] Конфігурації, початкові значення генераторів випадкових чисел, програмні залежності та необхідні артефакти збережено.
- [ ] Відхилення від протоколу задокументовано й за потреби погоджено з керівником.
- [ ] Код, результати пілоту, Implementation Report і README розміщено у власному GitHub fork.

**Статус власного проєкту:**

- `READY FOR EXPERIMENT` — реалізацію перевірено, пілот виконано, критичних невирішених проблем немає.
- `REVISION NEEDED` — виявлено помилки або невідповідності протоколу, які необхідно усунути до основного експерименту.

Самоперевірка має передувати review науковим керівником.


## E2. Контрольні питання

1. Чим програмна реалізація експерименту відрізняється від Experimental Protocol?
2. Чому потрібно виконувати split до стандартизації?
3. Як `Pipeline` допомагає уникати leakage під час Cross-Validation?
4. За яким критерієм у нашому прикладі оптимізуємо гіперпараметри? Чому не за шумовою стійкістю?
5. Чому не можна використовувати Test Set для вибору конфігурації моделі?
6. Чим Baseline Model відрізняється від Clean Reference Performance?
7. У яких одиницях задано $\sigma$ та на якому етапі додаємо шум?
8. Чому `fitted_pipeline.predict(z_noisy)` некоректний, якщо `z_noisy` уже стандартизовано?
9. Що означають однакові seed та спільна реалізація шуму для парного порівняння?
10. Чому п'ять повторів шуму на одному Test Set не доводять узагальнюваність на інші датасети?
11. Які файли, параметри й версії необхідні для відтворення реалізації?
12. Як пілотне вимірювання перевіряє весь ланцюжок від генерації шуму до `D(m,σ)`?
13. Чому один пілот не дозволяє оцінити H1 і як уникнути подвійного підрахунку цього вимірювання у Sprint 5?

## G. Solutions to Training Exercises — відповіді для самоперевірки

### Вправа 1

Стандартизація всього датасету використовує середнє та стандартне відхилення, обчислені також за майбутніми тестовими прикладами. Спочатку відокремлюємо Training Set і Test Set, потім навчаємо `StandardScaler` лише на Training Set та застосовуємо його до Test Set. При Cross-Validation `Pipeline` заново навчає scaler всередині кожної навчальної складки.

### Вправа 2

Виклик `fitted_pipeline.predict(z_noisy)` повторно застосує `StandardScaler`. Правильна послідовність: `raw Test Set → fitted scaler.transform → додавання ε → fitted classifier.predict`. Величина $\sigma$ задана для стандартизованих ознак, отже додавання шуму до сирих фізичних величин змінить зміст досліджуваного впливу.

### Вправа 3

Без фіксованих seed немає точної відтворюваності реалізацій; різний шум у моделях перешкоджає парному порівнянню. Заздалегідь визначаємо п'ять seed, генеруємо одну матрицю шуму для кожної пари ($\sigma$, seed) та застосовуємо її до тестових ознак усіх трьох моделей. Це вимірює варіативність впливу шуму на зафіксованому Test Set, але не перевіряє узагальнюваність на незалежних наборах даних.

## Підсумок і перехід до Sprint 5

**Sprint 1:** визначаємо Research Problem, H1 та Research Questions.

**Sprint 2:** критично аналізуємо джерела, уточнюємо H1 і створюємо Research Proposal.

**Sprint 3:** визначаємо Experimental Matrix, контрольовані умови, повторення та межі висновків.

**Sprint 4:** реалізуємо відтворюваний baseline, фіксуємо моделі й конфігурації та виконуємо пілотне вимірювання `Q` і `D`.

**Результат Sprint 4:** Reproducible Baseline v1.0 із перевіреною програмною реалізацією та документованим пілотним вимірюванням власного дослідження.

**Sprint 5:** виконуємо основні вимірювання за узгодженою матрицею, аналізуємо `Q(m,σ)`, `D(m,σ)` та невизначеність і повертаємося до RQ1–RQ3.
